In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_score, recall_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import json

In [ ]:
# -------- CONFIG --------
MODEL_NAME = "Maltehb/danish-bert-botxo"
TRAIN_PATH = "data/processed/train.parquet"
VAL_PATH = "data/processed/val.parquet"
MODEL_BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models"
SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/all_results.json"

MAX_LENGTHS = [256, 384, 512]
BATCH_SIZES = [4, 6, 8]
EPOCHS_LIST = [5, 10]
MODES = ["head", "mid", "tail"]

In [ ]:
# -------- LOAD DATA --------
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)

# define target column
label_cols = sorted([c for c in train_df.columns if c.startswith("label_")])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# -------- DATASET --------
class ClimateDataset(Dataset):
    def __init__(self, df, tokenizer, label_cols, max_len, mode="head"):
        self.texts = df["ArticleText"].tolist()
        self.labels = df[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.mode = mode

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        tokens = self.tokenizer(
            text,
            truncation=False,
            return_tensors="pt"
        )

        input_ids = tokens["input_ids"][0]

        # slicing
        if len(input_ids) > max_len:
            if self.mode == "head":
                input_ids = input_ids[:max_len]
            elif self.mode == "tail":
                input_ids = input_ids[-max_len:]
            elif self.mode == "mid":
                start = (len(input_ids) - max_len) // 2
                input_ids = input_ids[start:start + max_len]

        attention_mask = torch.ones(len(input_ids))

        # padding
        pad_len = max_len - len(input_ids)
        if pad_len > 0:
            input_ids = torch.cat([input_ids, torch.zeros(pad_len, dtype=torch.long)])
            attention_mask = torch.cat([attention_mask, torch.zeros(pad_len)])

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }


In [ ]:
# -------- METRICS --------
def multilabel_balanced_accuracy(labels, preds):
    scores = [
        balanced_accuracy_score(labels[:, i], preds[:, i])
        for i in range(labels.shape[1])
    ]
    return np.mean(scores)


def compute_all_metrics(labels, preds):
    return {
        "accuracy": multilabel_balanced_accuracy(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="micro", zero_division=0),
        "recall": recall_score(labels, preds, average="micro", zero_division=0),
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    return compute_all_metrics(labels, preds)


# -------- FINAL EVALUATION --------
def evaluate_model(trainer, dataset):
    output = trainer.predict(dataset)

    logits = output.predictions
    labels = output.label_ids

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    metrics = compute_all_metrics(labels, preds)
    metrics["loss"] = output.metrics["test_loss"]

    return metrics

In [ ]:
all_results = []

for max_len in MAX_LENGTHS:
    for batch_size in BATCH_SIZES:
        for epochs in EPOCHS_LIST:
            for mode in MODES:

                print(f"\n--- {mode} | max_len={max_len} | bs={batch_size} | ep={epochs} ---")

                # -------- DATA --------
                train_dataset = ClimateDataset(train_df, tokenizer, label_cols, max_len, mode)
                val_dataset   = ClimateDataset(val_df, tokenizer, label_cols, max_len, mode)

                # -------- MODEL --------
                model = AutoModelForSequenceClassification.from_pretrained(
                    MODEL_NAME,
                    num_labels=len(label_cols),
                    problem_type="multi_label_classification"
                )

                output_dir = f"{MODEL_BASE_PATH}/{mode}_ml{max_len}_bs{batch_size}_ep{epochs}"

                # -------- TRAINING ARGS --------
                training_args = TrainingArguments(
                    output_dir=output_dir,
                    per_device_train_batch_size=batch_size,
                    per_device_eval_batch_size=batch_size,
                    num_train_epochs=epochs,

                    eval_strategy="epoch",
                    save_strategy="epoch",
                    logging_strategy="epoch",

                    load_best_model_at_end=True,
                    metric_for_best_model="f1_macro",
                    greater_is_better=True,

                    save_total_limit=2
                )

                # -------- TRAINER --------
                trainer = Trainer(
                    model=model,
                    args=training_args,
                    train_dataset=train_dataset,
                    eval_dataset=val_dataset,
                    compute_metrics=compute_metrics,
                )

                # -------- TRAIN --------
                trainer.train()

                # -------- FINAL EVAL --------
                results = evaluate_model(trainer, val_dataset)

                results.update({
                    "mode": mode,
                    "max_len": max_len,
                    "batch_size": batch_size,
                    "epochs": epochs
                })

                all_results.append(results)

                # -------- SAVE RESULTS --------
                tmp_path = SAVE_PATH + ".tmp"

                with open(tmp_path, "w") as f:
                    json.dump(all_results, f, indent=2)

                os.replace(tmp_path, SAVE_PATH)

                print("Results saved")

In [ ]:
# save results
df = pd.DataFrame(all_results)
df.to_csv("grid_results.csv", index=False)